# 02 训练集与验证集：怎样判断模型真的学会了

上一课学会了读取标签，也讨论了翻转时如何同步改变框。这一课继续解决正式微调前的问题：**哪些图片用来学习，哪些图片用来检查学习效果？**

你已经在 MNIST 和 CNN 中接触过训练集、验证集。这里沿用相同原理，但要特别注意检测任务中的多目标和图片来源。

本课只理解划分原则，不运行数据准备或训练。

## 1. 先区分两种看图方式

同一套网络，在训练和验证时承担不同任务。

| 环节 | 训练集 | 验证集 |
|---|---|---|
| 图片是否进入模型 | 是 | 是 |
| 是否读取人工标签 | 是，用于监督 | 是，用于比较预测与参考答案 |
| 是否进行反向传播 | 是 | 否 |
| 是否执行优化器更新 | 是 | 否 |
| 主要目的 | 让参数从数据中学习 | 检查未参与该轮参数更新的数据上的表现 |

验证也需要正确答案，否则无法计算 Precision、Recall 和 mAP；区别在于，不利用验证数据执行反向传播和优化器更新。

验证时通常设置评估模式，并关闭梯度记录。评估模式还会改变 Dropout、BatchNorm 等层的行为；它与“不计算梯度”是两个不同的设置。

验证结果会间接影响我们选择哪个模型、如何调整超参数。因此，反复根据验证集做选择后，仍需要独立测试数据来支持最后的效果结论。

## 2. 为什么不能只看训练效果

假设学生反复练习同一套题，最后得了高分。这说明他能做这些题，还不能充分说明他能解决新题。

模型也可能记住训练图片的背景、人物或拍摄条件。训练集表现好，可能是学到了通用规律，也可能包含对这些图片的过度适应。

训练集与验证集表现的差距，可以帮助我们识别过拟合。验证集也只是对其所代表的数据分布提供有限估计，并不能保证任何摄像头场景都能做好。

例如，网络图片上的口罩检测表现好，不代表在教室监控中的远距离小人脸上同样好。

## 3. 本地的 853 张图片会怎样划分

已下载的压缩包中有 853 张 PNG 和 853 个同名 XML。本地旧脚本设置 `test_size=0.2`，以图片为单位随机划分，固定 `random_state=42`。

按该划分比例，会得到：

| 子集 | 图片数量 | 用途 |
|---|---:|---|
| train | 682 | 用于参数学习 |
| valid | 171 | 用于训练过程中的评估与模型选择 |

这是对现有脚本划分方式的说明，不是已经执行的结果。当前还没有生成完整的训练目录。

每张图片必须和它的标签一起分配。例如，第一课那张图片包含三个框，这三个框全部跟随该图片进入同一个子集，不能把同一图片中的不同人脸拆给训练集和验证集。

固定随机种子有助于复现划分，但不能保证没有重复或相似图片泄漏，也不能自动保证类别比例合理。

## 4. 先划分原始图片，再做训练增强

假设第一课那张图片叫 A，左右翻转后叫 A_flip。

如果把 A 放入训练集，把 A_flip 放入验证集，模型虽然没见过完全相同的像素排列，却已经接触过同一场景和人物。这份验证结果就不够独立。

正确顺序是：先按原始图片及其来源划分子集，再仅对训练样本进行随机增强。同一原图的增强版本应当跟随原图归属。

验证时仍然需要正常的尺寸调整、必要的补边和输入归一化；通常不采用训练中的随机翻转、随机裁剪或 Mosaic，以保持评估稳定、便于比较。

**训练数据增强的目标是帮助学习，不是制造看起来独立的新验证数据。**

## 5. 不同文件，也可能来自同一个场景

以下情况需要进一步按来源分组，而不是简单按文件名随机分配：

- 同一张图片的不同裁剪、不同分辨率版本或重复下载版本。
- 同一视频中相邻的帧。
- 同一次拍摄中几乎相同的连拍照片。

例如，要检验模型能否适应新视频，可以把同一段视频的帧放在同一子集，按视频或拍摄场次划分。若目标是适应全新的摄像头场景，还应考虑按摄像头或场景划分。

划分单位取决于我们希望验证什么泛化能力。当前本地脚本只做随机图片划分，尚未进行上述来源与相似图片检查。

## 6. 还要检查每类是否有足够的目标

直接读取当前压缩包的 XML，得到以下人工标注目标数量：

| 类别 | 标注框数 |
|---|---:|
| with_mask | 3232 |
| without_mask | 717 |
| mask_weared_incorrect | 123 |

这些是目标数，不是图片数；一张图片可以贡献多个目标，甚至包含多个类别。

佩戴不正确的目标明显较少。随机划分后，需要分别统计训练集与验证集中的类别分布，确认稀少类别有合理覆盖。

不能只看一个总体指标就声称三类都很好。后续还要看每类指标，并回到具体的漏检和误检图片上分析。

## 7. 为什么作者权重需要特别注意

当前网页加载的是作者提供的口罩权重，作者说明训练使用了我们下载的这个数据集。

我们尚未确认该权重具体训练过哪些图片、采用了哪份划分。因此，不能把它与我们重新随机划出的 171 张图片之间的关系，默认成“从未见过”。

如果用已有口罩权重继续训练，独立的新图片可以帮助评估这次微调是否改善了目标场景。

如果要从通用检测预训练权重开始完成一轮口罩任务训练，应先固定口罩数据的划分，并检查重复、来源相关性和上游预训练数据是否存在已知重叠。

本课没有选择或下载新的训练起点，也没有启动微调。

## 8. 用一句话检查是否理解

训练集参与反向传播和参数更新；验证集用于评估及模型选择；独立测试集用于在最终方案确定后评估其表现。

第一轮学习可以先理解训练集与验证集，不必现在就决定三个子集的精确比例。关键是先明确图片来源和各自用途。

请思考：

1. 验证集需要人工标注吗？为什么？
2. 把原图放进训练集、翻转图放进验证集，有什么问题？
3. 一段视频抽出 100 帧，随机分成 80 帧训练、20 帧验证，能否充分证明模型适应新视频？
4. `random_state=42` 能否保证验证结果可靠？

<details>
<summary>参考答案</summary>

1. 需要。要将模型预测与参考答案匹配，才能计算检测指标，但不据此进行反向传播和优化器更新。
2. 二者源自同一张图片，验证集与训练集不够独立，容易得到过于乐观的结果。
3. 不能。相邻帧可能高度相似；要评估新视频，应优先按视频或拍摄来源划分。
4. 不能。固定种子帮助复现，不能替代来源检查、去重和类别分布检查。

</details>

下一课：预训练权重、微调和从头训练有什么区别；加载旧权重后，究竟更新哪些参数。

## 本课依据与状态

- 图片数量和类别框数：直接统计本地项目 `datasets/archive.zip` 中的 PNG、XML 与 object/name。
- 随机划分方式：本地 `scripts/prepare_data.py` 中的 `train_test_split`。
- 现有权重与数据集关系：项目作者的训练说明；具体权重见过哪些图片尚未核实。

本课只新增笔记，没有解压完整数据集，没有执行正式划分，也没有启动训练。